In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../exports/cleaned_data/customers_clean.csv")

orders = pd.read_csv("../exports/cleaned_data/orders_clean.csv")

order_items = pd.read_csv("../exports/cleaned_data/order_items_clean.csv")

products = pd.read_csv("../exports/cleaned_data/products_clean.csv")

inventory = pd.read_csv("../exports/cleaned_data/inventory_clean.csv")

inventory_new = pd.read_csv("../exports/cleaned_data/inventory_new_clean.csv")

delivery = pd.read_csv("../exports/cleaned_data/delivery_clean.csv")

feedback = pd.read_csv("../exports/cleaned_data/feedback_clean.csv")

marketing = pd.read_csv("../exports/cleaned_data/marketing_clean.csv")

In [2]:
customer_features = (
    orders.groupby("customer_id")
    .agg({
        "order_total":["sum","mean","count"]
    })
)

In [3]:
customer_features.columns = [
    "total_revenue",
    "avg_order_value",
    "order_count"
]

In [4]:
customer_features = (
    customer_features
    .reset_index()
)

In [5]:
customer_features = (
    customer_features
    .merge(
        customers,
        on="customer_id",
        how="left"
    )
)

In [6]:
customer_features["revenue_per_order"] = (
    customer_features["total_revenue"]
    /
    customer_features["order_count"]
)

In [7]:
customer_features["value_score"] = (
    customer_features["total_revenue"]
    *
    customer_features["order_count"]
)

In [8]:
product_features = (
    order_items
    .merge(
        products,
        on="product_id",
        how="left"
    )
)

In [9]:
product_features["revenue"] = (
    product_features["quantity"]
    *
    product_features["unit_price"]
)

In [10]:
product_features = (
    product_features
    .groupby(
        [
            "product_id",
            "product_name",
            "category",
            "brand"
        ]
    )
    .agg({
        "revenue":"sum",
        "quantity":"sum"
    })
    .reset_index()
)

In [11]:
product_features["avg_selling_price"] = (
    product_features["revenue"]
    /
    product_features["quantity"]
)

In [12]:
inventory_features = (
    inventory
    .merge(
        products,
        on="product_id",
        how="left"
    )
)

In [13]:
inventory_features["damage_rate"] = (
    inventory_features["damaged_stock"]
    /
    inventory_features["stock_received"]
)

In [14]:
inventory_features["damage_rate"] = (
    inventory_features["damage_rate"]
    .fillna(0)
)

In [15]:
inventory_features = (
    inventory_features.groupby(
        [
            "product_id",
            "product_name",
            "category"
        ]
    )
    .agg({
        "stock_received":"sum",
        "damaged_stock":"sum",
        "damage_rate":"mean"
    })
    .reset_index()
)

In [16]:
orders["promised_delivery_time"] = pd.to_datetime(
    orders["promised_delivery_time"]
)

orders["actual_delivery_time"] = pd.to_datetime(
    orders["actual_delivery_time"]
)

In [17]:
orders["delivery_delay_minutes"] = (
    orders["actual_delivery_time"]
    -
    orders["promised_delivery_time"]
).dt.total_seconds() / 60

In [18]:
delivery_features = (
    orders.groupby("customer_id")
    .agg({
        "delivery_delay_minutes":"mean"
    })
    .reset_index()
)

In [19]:
marketing_features = (
    marketing.groupby("channel")
    .agg({
        "spend":"sum",
        "revenue_generated":"sum",
        "conversions":"sum",
        "roas":"mean"
    })
    .reset_index()
)

In [20]:
marketing_features[
    "conversion_efficiency"
] = (
    marketing_features["conversions"]
    /
    marketing_features["spend"]
)

In [21]:
kpis = {
    "Total Revenue":
    orders["order_total"].sum(),

    "Total Orders":
    len(orders),

    "Total Customers":
    customers["customer_id"].nunique(),

    "Average Order Value":
    orders["order_total"].mean(),

    "Average Rating":
    feedback["rating"].mean(),

    "Average ROAS":
    marketing["roas"].mean()
}

In [22]:
kpi_table = pd.DataFrame(
    kpis.items(),
    columns=[
        "KPI",
        "Value"
    ]
)

In [23]:
output_path = "../exports/processed_data/"

In [24]:
customer_features.to_csv(
    output_path + "customer_features.csv",
    index=False
)

product_features.to_csv(
    output_path + "product_features.csv",
    index=False
)

inventory_features.to_csv(
    output_path + "inventory_features.csv",
    index=False
)

delivery_features.to_csv(
    output_path + "delivery_features.csv",
    index=False
)

marketing_features.to_csv(
    output_path + "marketing_features.csv",
    index=False
)

kpi_table.to_csv(
    output_path + "master_kpis.csv",
    index=False
)